In [ ]:

from __future__ import annotations

import torch
from peft import PeftModel
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from src import ingest
from src.config import loadConfig, setSeeds

N_PAIRS = 25  # enough to be convincing, small enough to finish in a minute




In [ ]:
def main() -> None:
    config = loadConfig()
    setSeeds(config["seed"])

    baseName = config["matcher"]["lora"]["baseModel"]
    adapterPath = config["paths"]["loraAdapter"]
    print(f"base:    {baseName}")
    print(f"adapter: {adapterPath}\n")

    tokenizer = AutoTokenizer.from_pretrained(baseName)
    baseModel = AutoModelForSequenceClassification.from_pretrained(baseName, num_labels=3)
    model = PeftModel.from_pretrained(baseModel, adapterPath)
    model.eval()

    # Sanity check #1: did any LoRA weights actually load, and are they nonzero?
    # A freshly-initialised lora_B is all zeros -> the adapter is mathematically
    # the identity, no matter how correctly it's wired up.
    loraParams = [(n, p) for n, p in model.named_parameters() if "lora_" in n]
    nonzero = sum(1 for _, p in loraParams if p.abs().sum().item() > 0)
    print(f"lora tensors found: {len(loraParams)}  |  nonzero: {nonzero}")
    if loraParams:
        biggest = max(p.abs().max().item() for _, p in loraParams)
        print(f"largest single lora weight: {biggest:.6f}")
    if not loraParams:
        print("\n>>> VERDICT: no lora tensors at all. The adapter never loaded.")
        return
    if nonzero == 0:
        print("\n>>> VERDICT: every lora tensor is zero. The adapter is the identity.")
        return

    # Sanity check #2: the one that actually matters. Same input, adapter on vs
    # off, using peft's own switch so we compare the exact same model object.
    pairs = ingest.splitPairs(ingest.toEvalPairs(ingest.loadAnnotations()), config)["val"]
    pairs = [p for p in pairs if p.patientSpan][:N_PAIRS]
    print(f"\ncomparing on {len(pairs)} val pairs (gold spans)\n")

    maxDiff = 0.0
    flips = 0
    for pair in pairs:
        encoded = tokenizer(
            pair.patientSpan,
            pair.criterionText,
            truncation=True,
            max_length=config["matcher"]["lora"]["maxLength"],
            return_tensors="pt",
        )
        with torch.no_grad():
            withAdapter = model(**encoded).logits[0]
            with model.disable_adapter():
                withoutAdapter = model(**encoded).logits[0]

        maxDiff = max(maxDiff, (withAdapter - withoutAdapter).abs().max().item())
        if withAdapter.argmax().item() != withoutAdapter.argmax().item():
            flips += 1

    print(f"largest logit shift caused by the adapter: {maxDiff:.6f}")
    print(f"predictions the adapter flipped: {flips}/{len(pairs)}")

    if maxDiff < 1e-4:
        print("\n>>> VERDICT: adapter is a no-op. This is a WIRING bug.")
        print("    Fix the plumbing before touching lr, epochs, or the data.")
    elif flips == 0:
        print("\n>>> VERDICT: adapter is wired up but far too weak to change any answer.")
        print("    Cause C is real. Turn the volume up (after cleaning the data).")
    else:
        print("\n>>> VERDICT: adapter is alive and changing answers.")
        print("    It's a teaching problem, not a wiring problem. Clean the data first.")




In [ ]:
if __name__ == "__main__":
    main()